# Notebook 1: The Cooling Law — Why $p=2$ is the Unique Marginal Exponent

**Paper reference**: §II.D, Proposition 1 (Adiabatic cooling law)

**Goal**: Demonstrate that $\epsilon(n) = k/\ln^p(n)$ with $p=2$ is the unique marginal cooling schedule for a non-autonomous symplectic map approaching the Feigenbaum accumulation point.

**Key result**: For $p<2$, the time-averaged Lyapunov exponent $\bar\lambda \not\to 0$; for $p>2$, the system freezes prematurely; $p=2$ is the unique borderline.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Feigenbaum constants
mu_inf = 3.569945671870944   # accumulation point
delta_F = 4.669201609102990  # Feigenbaum delta
alpha_F = 2.502907875095892  # Feigenbaum alpha
k_opt = 1.248                # optimal cooling constant

print(f"μ_∞ = {mu_inf}")
print(f"δ_F = {delta_F}")
print(f"α_F = {alpha_F}")
print(f"k_opt = {k_opt}")

## 1.1 The Lyapunov exponent near the Feigenbaum point

Near $\mu_\infty$, the Lyapunov exponent of the logistic map scales as:
$$\lambda(\mu) \sim A\sqrt{\mu - \mu_\infty}$$
where $A \approx \ln 2$ for the logistic universality class.

In [ ]:
def logistic_lyapunov(mu, N_iter=50000, N_transient=5000):
    """Compute Lyapunov exponent of logistic map x → μx(1-x)."""
    x = 0.5 + 0.01 * np.random.randn()
    for _ in range(N_transient):
        x = mu * x * (1 - x)
        if x < 0 or x > 1: x = 0.5
    lyap = 0.0
    for _ in range(N_iter):
        deriv = abs(mu * (1 - 2*x))
        if deriv > 1e-15:
            lyap += np.log(deriv)
        x = mu * x * (1 - x)
        if x < 0 or x > 1: x = 0.5
    return lyap / N_iter

# Scan μ near μ_∞
mu_values = np.linspace(mu_inf + 1e-6, mu_inf + 0.1, 200)
lyap_values = [logistic_lyapunov(mu) for mu in mu_values]
eps_values = mu_values - mu_inf

# Fit λ = A√ε
from scipy.optimize import curve_fit
def sqrt_model(eps, A, B):
    return A * np.sqrt(eps) + B

mask = np.array(lyap_values) > 0
popt, _ = curve_fit(sqrt_model, eps_values[mask], np.array(lyap_values)[mask], p0=[0.5, 0])
print(f"Fitted: λ = {popt[0]:.4f}√ε + {popt[1]:.4f}")
print(f"Expected A ≈ ln(2) = {np.log(2):.4f}")

plt.figure(figsize=(8, 4))
plt.scatter(np.sqrt(eps_values), lyap_values, s=5, alpha=0.5, label='Simulation')
eps_fit = np.linspace(0, np.sqrt(eps_values.max()), 100)
plt.plot(eps_fit, popt[0]*eps_fit + popt[1], 'r-', lw=2, label=f'Fit: A={popt[0]:.3f}')
plt.xlabel(r'$\sqrt{\mu - \mu_\infty}$')
plt.ylabel(r'$\lambda$ (Lyapunov exponent)')
plt.title('Lyapunov scaling near the Feigenbaum point')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 1.2 Marginality of $p=2$

For cooling $\epsilon(n) = k/\ln^p(n)$, the time-averaged Lyapunov exponent is:
$$\bar\lambda(N) = \frac{1}{N}\sum_{n=1}^N A\sqrt{\frac{k}{\ln^p(n)}} = \frac{A\sqrt{k}}{N}\sum_{n=1}^N \frac{1}{\ln^{p/2}(n)}$$

- $p=1$: $\bar\lambda \sim 1/\sqrt{\ln N} \not\to 0$ ❌
- $p=2$: $\bar\lambda \sim 1/\ln N \to 0$ ✓ (but slowly!)
- $p=3$: $\bar\lambda \to 0$ but $\sum\sqrt{\epsilon}$ converges → system freezes ❌

In [ ]:
# Simulate cooling with different p values
A = np.log(2)
k = k_opt

N_max = 100000
ns = np.arange(3, N_max + 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, p in enumerate([1.0, 2.0, 3.0]):
    epsilon_n = k / np.log(ns)**p
    lambda_n = A * np.sqrt(epsilon_n)
    
    # Running average
    cumsum = np.cumsum(lambda_n)
    avg_lambda = cumsum / np.arange(1, len(lambda_n)+1)
    
    # Cumulative perturbation
    cum_sqrt_eps = np.cumsum(np.sqrt(epsilon_n))
    
    ax = axes[i]
    ax.semilogx(ns, avg_lambda, 'b-', lw=1, label=r'$\bar{\lambda}(N)$')
    ax.axhline(0, color='gray', ls='--', lw=0.5)
    ax2 = ax.twinx()
    ax2.semilogx(ns, cum_sqrt_eps, 'r-', lw=1, alpha=0.5, label=r'$\sum\sqrt{\epsilon}$')
    ax.set_title(f'p = {p}')
    ax.set_xlabel('N')
    ax.set_ylabel(r'$\bar{\lambda}$', color='b')
    ax2.set_ylabel(r'$\sum\sqrt{\epsilon}$', color='r')
    
    if p == 1:
        ax.set_title(f'p = {p}: $\\bar{{\\lambda}} \\not\\to 0$ ❌')
    elif p == 2:
        ax.set_title(f'p = {p}: marginal ✓')
    else:
        ax.set_title(f'p = {p}: freezes ❌')

plt.tight_layout()
plt.suptitle('Marginality of p=2 in the cooling law', y=1.02, fontsize=13)
plt.show()

## 1.3 Non-autonomous logistic map simulation

Run the actual logistic map with cooling $\mu(n) = \mu_\infty + k/\ln^p(n)$ and measure the time-averaged Lyapunov exponent for $p \in [1, 3]$.

In [ ]:
def simulate_cooling(p, k=k_opt, N=50000):
    """Simulate non-autonomous logistic map with cooling exponent p."""
    x = 0.5
    lyap_sum = 0.0
    for n in range(3, N + 3):
        mu_n = mu_inf + k / np.log(n)**p
        deriv = abs(mu_n * (1 - 2*x))
        if deriv > 1e-15:
            lyap_sum += np.log(deriv)
        x = mu_n * x * (1 - x)
        if x < 0 or x > 1:
            x = 0.5
    return lyap_sum / N

p_values = np.linspace(1.0, 3.0, 50)
avg_lyap = [simulate_cooling(p) for p in p_values]

plt.figure(figsize=(8, 4))
plt.plot(p_values, avg_lyap, 'bo-', ms=4)
plt.axhline(0, color='r', ls='--', lw=1)
plt.axvline(2.0, color='green', ls=':', lw=1, label='p = 2 (marginal)')
plt.xlabel('Cooling exponent p')
plt.ylabel(r'$\bar{\lambda}$ (time-averaged Lyapunov)')
plt.title('Time-averaged Lyapunov exponent vs cooling exponent p')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Find the zero crossing
for i in range(len(avg_lyap)-1):
    if avg_lyap[i] > 0 and avg_lyap[i+1] <= 0:
        p_cross = p_values[i] + (p_values[i+1]-p_values[i]) * avg_lyap[i]/(avg_lyap[i]-avg_lyap[i+1])
        print(f"Zero crossing at p ≈ {p_cross:.3f}")
        break

## Conclusion

The cooling law $\mu(n) = \mu_\infty + k/\ln^2(n)$ is the **unique marginal schedule** that:
1. Drives $\bar\lambda \to 0$ (approach criticality)
2. Keeps $\sum\sqrt{\epsilon} \to \infty$ (no premature freezing)

This is an **asymptotic** result valid under saddle-node scaling $\lambda \sim \sqrt{\mu - \mu_c}$.